[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://github.com/codespaces)


<!-- gen:values.yml#weeks.02b.notebook-title|Debrief -->
# Week 2b Debrief: Advanced Python
<!-- /gen -->

*Worked answers for [Quest](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-quest) 02b, alternative approaches, and the [AI failure gallery](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-ai-failure-gallery) of ways AI tools plausibly get this week's material wrong.*

**Read this only after attempting the [quest](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-quest).** If you have not, close this and open [Quest.ipynb](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Week/02b/Quest.ipynb).

The data-setup cell below is identical to the quest's (same seed, same 320 subscribers), so every number here should match yours.

In [ ]:
# --- Data setup: Run this cell first (runs anywhere, no download needed) ---
import numpy as np

rng = np.random.default_rng(5813)  # module seed: Do not change

# StreamShore launched in January 2025. Months are numbered from launch:
# month 0 = January 2025, so "today" (June 2026) is month 17.
TODAY = 17

# An alternative way to format a dictionary,
# which may be easier to read ...
PLAN_FEES = {
    "Basic": 5.99,
    "Standard": 9.99,
    "Premium": 13.99,
}
CHURN_RISK = {"Basic": 0.055, "Standard": 0.038, "Premium": 0.022}
PLAN_MIX = ["Basic"] * 24 + ["Standard"] * 54 + ["Premium"] * 22

subscribers = []

# App sign-ups: Joined at some point between launch and month 14.
for i in range(276):
    plan = PLAN_MIX[int(rng.integers(0, 100))]
    signup = int(rng.integers(0, 15))
    months_kept = int(rng.geometric(CHURN_RISK[plan])) - 1
    cancelled = signup + months_kept
    if cancelled > TODAY:
        cancelled = None   # cancellation month hasn't arrived yet
    subscribers.append({
        "id": f"SS-{1000 + i}",
        "plan": plan,
        "signup_month": signup,
        "cancelled_month": cancelled,   # None = still subscribed today
        "monthly_fee": PLAN_FEES[plan],
    })

# Founding members: Premium at the £11.99 launch offer, imported from the
# old billing spreadsheet, which had its own ideas about column names.
for i in range(34):
    months_kept = int(rng.geometric(0.018)) - 1
    cancelled = months_kept
    if cancelled > TODAY:
        cancelled = None
    subscribers.append({
        "id": f"SS-{9000 + i}",
        "plan": "Premium",
        "signup_month": 0,
        "cancelled_month": cancelled,
        "monthly_price": 11.99,
    })

# Launch-week trialists: Grabbed the free first month, left before paying.
for i in range(10):
    subscribers.append({
        "id": f"SS-{9500 + i}",
        "plan": "Basic",
        "signup_month": 0,
        "cancelled_month": 0,
        "monthly_fee": PLAN_FEES["Basic"],
    })

rng.shuffle(subscribers)

active = sum(1 for s in subscribers if s["cancelled_month"] is None)
print(f"Created subscribers: {len(subscribers)} records "
      f"({active} active, {len(subscribers) - active} cancelled)")

## 🔍 First look: What you should have found

- **[First look 1](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-first-look)**: 320 records, one dict per subscriber with the keys `id`, `plan`, `signup_month`, `cancelled_month`, and `monthly_fee`. Your first record is a still-active Premium subscriber from launch month, the same one for everyone because the shuffle is seeded
- **First look 2**: `{'Basic': 79, 'Standard': 156, 'Premium': 85}`. Standard is roughly half the base
- **First look 3**: `sorted` raises `KeyError: 'monthly_fee'`. It applies the `key` lambda to every record, and 34 of them do not have that key
- **First look 4**: Those 34 are the founding members. Their fee sits under `monthly_price`, at £11.99, and all have `signup_month = 0`

The cell below checks the plan counts again and describes the 34 founding-member records.

In [ ]:
# First look 2, verified, and the founding members characterised.
plan_counts = {plan: sum(1 for s in subscribers if s["plan"] == plan)
               for plan in ["Basic", "Standard", "Premium"]}
print(f"Plan counts: {plan_counts}  (sum: {sum(plan_counts.values())})")

legacy = [s for s in subscribers if "monthly_fee" not in s]
print(f"Legacy records: {len(legacy)}, plans: {set(s['plan'] for s in legacy)}, "
      f"fee key: monthly_price = £{legacy[0]['monthly_price']}")

Before you trust any field, **survey the keys**. The line `[s for s in data if "field" not in s]` lists every record in any dataset that is missing the field you name.

## 🥉 Bronze: Churn per plan, worked

The cell below uses one comprehension per quantity, as [the Demonstration](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Week/02b/Demonstration.ipynb) requires, and prints the reconciliation totals as it goes.

In [ ]:
print(f"{'Plan':10s} {'Members':>8s} {'Cancelled':>10s} {'Churn rate':>11s}")

total_members = 0
total_cancelled = 0
for plan in ["Basic", "Standard", "Premium"]:
    members = [s for s in subscribers if s["plan"] == plan]
    cancelled = [s for s in members if s["cancelled_month"] is not None]
    rate = len(cancelled) / len(members)
    total_members += len(members)
    total_cancelled += len(cancelled)
    print(f"{plan:10s} {len(members):8d} {len(cancelled):10d} {rate:10.1%}")

# The reconciliation. It is non-negotiable before any number reaches a memo:
print(f"\nReconcile: {total_members} members (expect 320), "
      f"{total_cancelled} cancelled (expect the data-setup total)")

**The numbers:**

- Basic: 79 members, 46 cancelled, churn rate **58.2%**
- Standard: 156 members, 51 cancelled, churn rate **32.7%**
- Premium: 85 members, 25 cancelled, churn rate **29.4%**

**The memo sentence:** *"Basic loses customers fastest: Nearly six in ten Basic subscribers have already cancelled. The 'Premium is our weakest plan' rumour is false. Premium has the best retention of the three plans."*

The trap in this data is the gap between counts and rates. Standard has the most cancellations (51), but **Basic has by far the worst rate**. If you report counts without rates, the board looks for the problem in the wrong plan. The same happens if you prompt an [Associate](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-associate) only for "cancellations per plan".

If your Associate wrote the whole table as one comprehension and you checked the totals, that answer is also fine.

## 🥈 Silver: The lifetime judgement, worked

The quest's draft averages **completed lifetimes for the 122 cancelled subscribers** together with **unfinished lifetimes for the 198 active ones**. It treats today as everyone's cancellation day. The result, 8.3 months, is not the average of any real quantity. It mixes "how long quitters lasted" with "how long stayers have lasted *so far*".

The cell below computes both fixes and labels each.

In [ ]:
# Fix A: Completed lifetimes only (subscribers who have cancelled).
completed = [s["cancelled_month"] - s["signup_month"]
             for s in subscribers if s["cancelled_month"] is not None]
fix_a = sum(completed) / len(completed)

# Fix B: Everyone, censored at today, reported as a lower bound.
so_far = [(s["cancelled_month"] if s["cancelled_month"] is not None else TODAY)
          - s["signup_month"]
          for s in subscribers]
fix_b = sum(so_far) / len(so_far)
still_active = sum(1 for s in subscribers if s["cancelled_month"] is None)

print(f"Fix A: cancelled subscribers lasted {fix_a:.1f} months on average "
      f"(n={len(completed)})")
print(f"Fix B: subscribers have stayed AT LEAST {fix_b:.1f} months on average; "
      f"{still_active} of {len(subscribers)} ({still_active/len(subscribers):.0%}) "
      f"are still active")

**4.3 months versus "at least 8.3"**: Changing the definition nearly *doubles* the headline number. You hand in the defence of the definition you choose, not the arithmetic.

**A defensible case for A:** The board asks how long a customer who cancels stays. Fix A is a real average of finished lifetimes, with no forecast hiding inside it. The 4.3-month figure says churn happens *early*, so the company can act on it by fixing onboarding and the month-3 drop-off. Its weakness is that it ignores the 62% who are still paying, so it makes the typical customer look much worse than they are. It also looks *worse* as the company grows, because each burst of new signups adds fast-quitting trialists to the sample.

**A defensible case for B:** The board is choosing which plan to *push*, so it needs observed loyalty. "At least 8.3 months and climbing, 62% still active" describes the whole base, not only its quitters. Its weakness is that the number keeps rising as time passes. It is a minimum, not a lifetime. Quote it without the "at least" and it becomes as false as the original draft.

**What a strong [Silver](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-silver) defence looks like:** It names the board's actual question and chooses the number that answers *that* question. It also states the chosen number's weakness out loud. Many strong answers choose *both*: A for the churn discussion, B for the loyalty headline. An answer that says no more than "B, because it uses all the data" misses the point.

The proper tool for this question is survival analysis, which comes in later modules. "Censoring" is the word to search for when you meet it.

## 🥇 Gold: One worked example, revenue retained by cohort

[Gold](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-gold) has no single right answer. Here is one solid version, with the fee-key lesson applied. `fee_of` uses a chain of [`.get()`](https://docs.python.org/3/library/stdtypes.html#dict.get) calls, so it accepts either fee key. If a record has neither key, a `raise` statement ([raising exceptions](https://docs.python.org/3/tutorial/errors.html#raising-exceptions)) crashes loudly on purpose.

In [ ]:
def fee_of(s):
    """Monthly fee, whichever key the record uses. Crashes loudly if neither."""
    fee = s.get("monthly_fee", s.get("monthly_price"))
    if fee is None:
        raise KeyError(f"{s['id']}: no fee under either key")
    return fee

print(f"{'Cohort':16s} {'Signed up':>9s} {'Still active':>13s} {'Revenue to date':>16s}")
for q_start in range(0, 15, 3):
    cohort = [s for s in subscribers
              if q_start <= s["signup_month"] < q_start + 3]
    active = [s for s in cohort if s["cancelled_month"] is None]
    revenue = sum(((s["cancelled_month"] if s["cancelled_month"] is not None
                    else TODAY) - s["signup_month"]) * fee_of(s)
                  for s in cohort)
    print(f"months {q_start:2d}-{q_start + 2:2d}     {len(cohort):9d} "
          f"{len(active)/len(cohort):13.0%} {revenue:15,.2f}")

**Reading it for the board:** The launch cohort's revenue is large mostly because it has had the longest to pay. The money column makes the oldest cohorts look best. The *retention* column holds the surprise. Cohorts from month 6 onwards keep **about 67% to 73% active**. The first two quarters keep about 52% to 56%. The memo sentence: *"Whatever StreamShore changed around month 6, it is working. Newer cohorts are retaining roughly 15 points better than launch-era ones."* That is a sentence the board does not already suspect, which is what Gold asks for.

Two strong alternatives:

- **The trial-leak analysis**: The 10 launch trialists plus every other first-month quitter, priced per month in lost fees
- **A fee-weighted churn table**: Which plan loses the most **money** per month, not the most customers

## 🚩 Red Flags: The silent drop, worked

**The bug.** The founding members carry their fee under `monthly_price`, so `s["monthly_fee"]` raises `KeyError` for all 34 of them. The line `except KeyError: pass` throws each one away without a word. `len(processed)` is 286, not 320.

**In business terms.** The dropped records are not a random sample. They are *every founding member*, all Premium, all paying since launch, and the most valuable block of customers StreamShore has. The draft understates Premium revenue by roughly half. About £6,200 of Premium money vanishes, so the board sees "Standard £12.4k, Premium far behind at £6.3k" and concludes Premium underperforms. On the fixed figures Premium is StreamShore's **top-earning plan**. On the draft, Friday's board would put the budget into Standard and take it away from StreamShore's best plan.

**The corrected version** below accepts both keys, counts the fallback, and keeps the loud failure for genuinely broken records.

In [ ]:
revenue_by_plan = {}
used_fallback = 0

for s in subscribers:
    end = s["cancelled_month"] if s["cancelled_month"] is not None else TODAY
    months_paid = end - s["signup_month"]
    try:
        fee = s["monthly_fee"]
    except KeyError:
        fee = s["monthly_price"]   # legacy key; if THIS also fails, crash loudly
        used_fallback += 1
    revenue_by_plan[s["plan"]] = (revenue_by_plan.get(s["plan"], 0)
                                  + months_paid * fee)

print("=== Revenue by plan (since launch, all records) ===")
for plan, total in sorted(revenue_by_plan.items(), key=lambda kv: kv[1],
                          reverse=True):
    print(f"{plan:9s} £{total:,.2f}")
print(f"\nRecords using the legacy fee key: {used_fallback} (expect 34)")

The ranking flips: **Premium £12,482 is now just ahead of Standard £12,407**. Premium does that from a third of the subscriber count. The `except KeyError` *handles* the known quirk and counts it. A record with neither key still crashes loudly.

**How to spot this anywhere:** Whenever code contains `except ... : pass` or `except ... : continue`, ask "how do I know how many records took this path?" If the code cannot answer, count the skips or reconcile the processed records against the input, *every time*.

## 🤖 AI failure gallery

Below are three failures AI tools plausibly <!-- plain-language: everyday commit -->commit on *this week's* material. Every one runs without an error and returns a wrong answer that looks right. For each failure you get the plausible output, why it is wrong in business terms, and the catch.

### Failure 1: The bare `except`, errors as landfill

Quest 02b's Red Flag swallowed a *known* exception. A fully bare `except:` is worse, because it also swallows exceptions nobody predicted, including typos in the Associate's own draft. Asked for "robust" monthly-revenue code, an Associate plausibly produces the cell below.

In [ ]:
# Plausible AI output: "robust" current monthly revenue (active subscribers).
def monthly_revenue(subs):
    total = 0
    for s in subs:
        try:
            if s["canceled_month"] is None:      # <- US spelling: Wrong key!
                total += s.get("monthly_fee", s.get("monthly_price", 0))
        except:
            continue   # skip anything problematic
    return total

print(f"Current monthly revenue: £{monthly_revenue(subscribers):,.2f}")

**Why it is wrong in business terms:** £0.00. StreamShore apparently has no revenue at all. The Associate typed `canceled_month` (US spelling), so *every* record raised `KeyError`, and the bare `except` turned a one-letter typo into "skip all 320 customers". Here the absurd total shows that something is wrong. A smaller typo hits only some records. The total then looks believable, and someone builds a cash-flow forecast on it.

**The catch:** *Any* bare `except` or `except: continue` in a draft is the warning sign. Delete it, re-run, and let the crash name the real problem. Here the traceback would say `KeyError: 'canceled_month'`, a diagnosis in one line. Handle the exception you expect, by name, and count what you skip.

### Failure 2: The mutable default argument, reports that haunt each other

This is a classic Python trap, because the code *looks* correct. Asked for "a helper that collects at-risk subscribers into a report list", the Associate offers the cell below.

In [ ]:
# Plausible AI output: Collect at-risk subscribers (early-quitters) per plan.
def at_risk_report(subs, plan, report=[]):    # <- the trap: report=[]
    for s in subs:
        cm = s["cancelled_month"]
        if s["plan"] == plan and cm is not None and cm - s["signup_month"] <= 1:
            report.append(s["id"])
    return report

basic_report = at_risk_report(subscribers, "Basic")
print(f"Basic early-quitters:   {len(basic_report)}")

premium_report = at_risk_report(subscribers, "Premium")
print(f"Premium early-quitters: {len(premium_report)}   <- read this twice")

**Why it is wrong in business terms:** The "Premium" report contains every *Basic* early-quitter too. Python evaluates a default argument **once**, when the function is defined, so `report=[]` is one shared list that every call appends to. The second report is contaminated by the first. Run the cell again and both numbers grow. Premium has the *best* retention in [Bronze](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-bronze), but this report gives it dozens of early quitters. At a firm like Runnymede, this is how one client's records end up in another client's deliverable.

**The catch:** Print the first few IDs in `premium_report` and look them up. Basic subscribers sit inside a report whose filter says `plan == "Premium"`, and re-running the cell grows numbers that should stay fixed. The pattern to demand instead: `def at_risk_report(subs, plan, report=None):`, then `if report is None: report = []` inside. Any `def` with `=[]` or `={}` in its signature is a red flag on sight.

### Failure 3: The comprehension that silently drops records

A comprehension with the wrong condition gives you the wrong records, and Python shows no error. Asked to "grab the cancelled subscribers", an Associate plausibly writes the first line in the cell below.

In [ ]:
# Plausible AI output: "the cancelled subscribers".
cancelled_v1 = [s for s in subscribers if s["cancelled_month"]]            # truthy
cancelled_v2 = [s for s in subscribers if s["cancelled_month"] is not None] # correct

print(f"Truthy filter:      {len(cancelled_v1)} cancelled")
print(f"'is not None':      {len(cancelled_v2)} cancelled")

dropped = [s for s in cancelled_v2 if s not in cancelled_v1]
print(f"Silently dropped:   {len(dropped)} (e.g. {dropped[0]['id']}, "
      f"plan {dropped[0]['plan']}, cancelled in month {dropped[0]['cancelled_month']})")

**Why it is wrong in business terms:** `if s["cancelled_month"]` treats `0` as false, so everyone who cancelled in **month 0 vanishes**. The first filter keeps 112 records. The `is not None` filter keeps 122. The ten missing records are the launch-month trialists, all first-month quitters. Those are the records that measure onboarding failure and trial abuse, so an analysis of "why do people quit early?" built on this filter has excluded its own subject.

**The catch:** Reconcile every filter. The records you keep plus the records you drop must equal the input. Then spot-check the boundary values, such as month 0, the earliest date, £0.00, and empty strings. Data that is valid but reads as false is where these bugs live. Fix the prompt too. Ask for a comparison with `is not None` when `None` stands for "no value", as the quest's Bronze prompt does.

**The common thread:** None of these three failures announces itself. The habits that catch them are the quest's habits: Reconcile counts, distrust silence, and read the condition. Apply them every time, especially when the draft "looks fine".

## 🚪 Closing the file

Update your `QuestLog.md` with the [badge](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-badge) you claim, and check that the [AI Use Record](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-ai-use-record) in the quest is honest.

**Where next:** Week 3 is the assessment briefing, and nothing in it depends on this optional material.